In [62]:
# ============================================================================
# 03a — SETUP: BENCHMARK, PARSER, MODELS, PROVIDER PINNING
# ----------------------------------------------------------------------------
# Five stages, all cheap:
#   1. load the benchmark, prompts and answer parser
#   2. ID CHECK   — one call per model, catches typos before they cost money
#   3. PIN PROBE  — for multi-provider models, find one that can serve the run
#   4. APPLY      — route each model to a fixed provider so a mid-run switch
#                   cannot change the answers
#   5. VERIFY     — confirm the pins hold
# Costs a few cents. Sends nothing large.
import os, json, time, re, threading, hashlib, requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pandas as pd, numpy as np
from dotenv import load_dotenv

ROOT = Path.cwd().parent
load_dotenv(ROOT/".env", override=True)
KEY = os.environ.get("OPENROUTER_API_KEY", "")
assert len(KEY) > 20, "OPENROUTER_API_KEY missing from .env"
ENDPOINT = "https://openrouter.ai/api/v1/chat/completions"

PROC, RES = ROOT/"data"/"processed", ROOT/"results"
RES.mkdir(parents=True, exist_ok=True)
VERSION = "v2"

# ============================ 1. BENCHMARK ================================
FULL  = pd.read_json(PROC/f"sighabit_{VERSION}_full.jsonl", lines=True)
USUAL = pd.read_json(PROC/f"sighabit_{VERSION}_usually.jsonl", lines=True)
PR    = json.loads((PROC/f"prompts_{VERSION}.json").read_text())
SHA   = hashlib.sha256((PROC/f"sighabit_{VERSION}_full.jsonl").read_bytes()).hexdigest()[:16]
MAIN_ONLY = FULL[~FULL.is_decoy].reset_index(drop=True)   # swap arm
WEB_ITEMS = MAIN_ONLY                                     # web arm (no decoys: search fees)
print(f"benchmark {len(FULL)} main+decoy | {len(USUAL)} usually | sha={SHA}")
print(f"  {dict(FULL.condition.value_counts())}")

def render(item, swap=False):
    if item.get("condition") == "usually":
        body = (f"Across its known direct targets, what is the net regulatory "
                f"effect of {item['source']} at the protein level?\n{PR['guide_usually']}")
    else:
        body = (f"What is the net regulatory effect of {item['source']} on "
                f"{item['target']} at the protein level?\n{PR['guide']}")
    return f"{body}\n\nRespond with JSON only:\n" + (
        PR['schema_swapped'] if swap else PR['schema'])

# ============================ PARSER ======================================
SYN = {"stimulates":"stimulates","stimulate":"stimulates","activates":"stimulates",
 "activate":"stimulates","activation":"stimulates","positive":"stimulates",
 "upregulates":"stimulates","induces":"stimulates",
 "inhibits":"inhibits","inhibit":"inhibits","inhibition":"inhibits",
 "represses":"inhibits","repress":"inhibits","negative":"inhibits",
 "downregulates":"inhibits","suppresses":"inhibits",
 "not_established":"not_established","not established":"not_established",
 "unknown":"not_established","none":"not_established","unclear":"not_established"}

def strip_reasoning(t):
    """Reasoning models restate the answer template in their scratchpad;
    strip it or we parse the example instead of the answer."""
    t = re.sub(r"<think>.*?</think>", " ", t, flags=re.S|re.I)
    t = re.sub(r"^.*?</think>", " ", t, flags=re.S|re.I)
    return t.strip() or t

def parse(text):
    if not isinstance(text,str) or not text.strip(): return None,"failed"
    b = strip_reasoning(text)
    try:
        o = json.loads(b)
        if isinstance(o,dict): return o,"strict"
    except Exception: pass
    c = re.sub(r"```(?:json)?|```", " ", b)
    for blob in reversed(re.findall(r"\{[^{}]*\}", c, flags=re.S)):
        try:
            o = json.loads(re.sub(r",\s*([}\]])", r"\1", blob))
            if isinstance(o,dict) and "answer" in o: return o,"repaired"
        except Exception: continue
    m = re.search(r'"?answer"?\s*[:=]\s*"?([A-Za-z_ ]+?)"?\s*[,\n}]', c, re.I)
    if m:
        out = {"answer": m.group(1).strip()}
        cf = re.search(r'"?confidence"?\s*[:=]\s*(\d+)', c, re.I)
        if cf: out["confidence"] = int(cf.group(1))
        return out,"regex"
    return None,"failed"

def normalize(p):
    if not p or "answer" not in p: return None
    return SYN.get(str(p["answer"]).strip().lower().replace("-","_"))

for txt, want in [('{"answer":"stimulates","confidence":8}',"stimulates"),
                  ('```json\n{"answer":"inhibits",}\n```',"inhibits"),
                  ('<think>maybe stimulates</think>{"answer":"inhibits"}',"inhibits"),
                  ('{"answer":"activates"}',"stimulates"),
                  ('{"answer":"not_established"}',"not_established"),
                  ('I think it activates.',None)]:
    assert normalize(parse(txt)[0]) == want, f"parser failed on {txt!r}"
print("parser self-test: 6/6 pass")

# ============================ MODELS ======================================
# name, id, $/M in, $/M out, max_tok, providers, service
# Empty provider list = single-vendor. service = "openrouter" or "hf".
MODELS_SPEC = [
 # ---- already collected: leave in so the code knows about them ----
 ("gpt-oss-120b",     "openai/gpt-oss-120b",               0.03,  0.17,  6000,
    ["DeepInfra","CoreWeave","Novita"], "openrouter"),
 ("qwen3-32b",        "qwen/qwen3-32b",                    0.10,  0.30,  4000,
    ["DeepInfra","Nebius","SiliconFlow"], "openrouter"),
 ("gemini-3.7-flash", "google/gemini-3.7-flash",           0.1875,0.9375,3000, [], "openrouter"),
 ("gpt-5.6-luna",     "openai/gpt-5.6-luna",               0.10,  0.60,  5000, [], "openrouter"),
 ("gpt-5.6-terra",    "openai/gpt-5.6-terra",              1.00,  6.00,  5000, [], "openrouter"),
 ("claude-haiku-4.5", "anthropic/claude-haiku-4.5",        1.00,  5.00,  4000, [], "openrouter"),
 ("llama-3.3-70b",    "meta-llama/llama-3.3-70b-instruct", 0.12,  0.30,  3000,
    ["DeepInfra","Novita","Nebius"], "openrouter"),
 ("mistral-large",    "mistralai/mistral-large",           2.00,  6.00,  3000, [], "openrouter"),
 ("deepseek-v3",      "deepseek/deepseek-chat",            0.14,  0.28,  4000,
    ["DeepInfra","Novita","Together"], "openrouter"),
 ("gemma-3-27b",      "google/gemma-3-27b-it",             0.10,  0.20,  3000,
    ["DeepInfra","Novita","Nebius"], "openrouter"),
 ("qwen3-235b",       "qwen/qwen3-235b-a22b",              0.20,  0.60,  4000,
    ["DeepInfra","Nebius","Together"], "openrouter"),

 # ---- NEW ----
 # size axis: everything above is 27B+; this is the small end
 ("llama-3.1-8b",     "meta-llama/llama-3.1-8b-instruct",  0.02,  0.03,  3000,
    ["DeepInfra","Novita","Nebius"], "openrouter"),
 # biomedical: MedGemma pairs against gemma-3-27b (same base, same size)

 ("medgemma-27b",     "google/medgemma-27b-text-it:featherless-ai",
                                                           0.10,  0.20,  3000, [], "hf"),
 # newest Gemma, alongside gemma-3-27b and medgemma-27b
 ("gemma-4-31b",      "google/gemma-4-31b-it",             0.12,  0.35,  3000,
    ["DeepInfra","Novita","Nebius","Together"], "openrouter"),
 # second biomedical pair: Med42-v2-70B is instruct+preference tuned from
 # Llama-3-70B, so these two share base, version and size exactly.
 ("llama-3-70b",      "meta-llama/Meta-Llama-3-70B-Instruct:featherless-ai",
                                                           0.30,  0.60,  2048, [], "hf"),
 ("med42-70b",        "m42-health/Llama3-Med42-70B:featherless-ai",
                                                           0.30,  0.60,  2048, [], "hf"),

]

# models that also get a web-search arm (retrieval currently rests on n=1)
WEB_MODELS = ["gemini-3.7-flash", "gpt-5.6-terra"]
BUDGET, WORKERS, TEMP = 2.00, 6, 0.3     # OpenRouter + HuggingFace headroom
SPEND = {"usd":0.0}; STOP = threading.Event(); LOCK = threading.Lock()
DEAD  = set()          # models blocked mid-run; skipped rather than halting all

def _post(body, timeout):
    r = requests.post(ENDPOINT, headers={"Authorization":f"Bearer {KEY}"},
                      json=body, timeout=timeout)
    return r



# ---- HuggingFace router, for models OpenRouter does not carry -------------
HF_KEY = os.environ.get("HF_TOKEN", "")
HF_ENDPOINT = "https://router.huggingface.co/v1/chat/completions"
HAVE_HF = len(HF_KEY) > 20
print(f"HuggingFace token: {'present' if HAVE_HF else 'MISSING — HF models will be skipped'}")

def _post_any(body, timeout, service="openrouter"):
    """Same call shape for both services — both speak the OpenAI format."""
    if service == "hf":
        return requests.post(HF_ENDPOINT,
                             headers={"Authorization": f"Bearer {HF_KEY}"},
                             json=body, timeout=timeout)
    return requests.post(ENDPOINT,
                         headers={"Authorization": f"Bearer {KEY}"},
                         json=body, timeout=timeout)

# ==================== 2. ID CHECK ========================================
print("\n"+"="*92); print("2. MODEL ID CHECK  (1 call each)"); print("="*92)
alive = []
for name, mid, pi, po, mt, cands, svc in MODELS_SPEC:
    if svc == "hf" and not HAVE_HF:
        print(f"  [SKIP] {name:<20}no HF_TOKEN in .env"); continue
    body = {"model":mid, "temperature":0, "max_tokens":32,
            "messages":[{"role":"user","content":"Reply with the single word: ok"}]}
    try:
        r = _post_any(body, 120, svc)
        if r.status_code == 200:
            d = r.json()
            txt = (d["choices"][0]["message"].get("content") or "").strip()[:18]
            print(f"  [OK ] {name:<20}{svc:<11}{mid[:44]:<46}{txt!r}")
            alive.append((name, mid, pi, po, mt, cands, svc))
        else:
            print(f"  [BAD] {name:<20}{svc:<11}{mid[:44]:<46}"
                  f"HTTP {r.status_code} {r.text[:60]}")
    except Exception as e:
        print(f"  [BAD] {name:<20}{svc:<11}{mid[:44]:<46}{type(e).__name__}")
dropped = [m[0] for m in MODELS_SPEC if m[0] not in {a[0] for a in alive}]
print(f"\n  usable {len(alive)}/{len(MODELS_SPEC)}"
      + (f"   DROPPED: {dropped}" if dropped else ""))

_MID2SVC = {mid: svc for _, mid, _, _, _, _, svc in alive}

# ==================== 3. PIN PROBE (OpenRouter multi-provider only) ======
print("\n"+"="*92); print("3. PROVIDER PROBE"); print("="*92)
PROBE_N = 6
probe_items = FULL.head(PROBE_N)
PIN = {}
for name, mid, pi, po, mt, cands, svc in alive:
    if svc == "hf":
        PIN[name] = None
        print(f"  {name:<20}HuggingFace — single endpoint, nothing to pin"); continue
    if not cands:
        PIN[name] = None
        print(f"  {name:<20}single-vendor — nothing to pin"); continue
    print(f"  {name}")
    for cand in cands:
        good, provs, t0 = 0, set(), time.time()
        for _, it in probe_items.iterrows():
            try:
                b = {"model":mid, "temperature":TEMP, "max_tokens":mt,
                     "messages":[{"role":"system","content":PR["system"]},
                                 {"role":"user","content":render(it.to_dict())}],
                     "provider":{"order":[cand], "allow_fallbacks":False}}
                r = _post_any(b, 120, "openrouter")
                if r.status_code == 200:
                    d = r.json()
                    good += bool(d["choices"][0]["message"].get("content"))
                    if d.get("provider"): provs.add(d["provider"])
            except Exception: pass
        honoured = (provs <= {cand}) or not provs
        print(f"    {cand:<14}{good}/{PROBE_N}  {good/PROBE_N:>5.0%}  "
              f"{(time.time()-t0)/PROBE_N:>5.1f}s  got={provs or '?'}"
              + ("" if honoured else "  *** PIN IGNORED ***"))
        if good/PROBE_N >= 0.83 and honoured:
            PIN[name] = cand; print(f"    --> PINNED to {cand}"); break
    else:
        PIN[name] = None; print(f"    --> unpinned (provider logged per call)")

# ==================== 4. APPLY ===========================================
MODELS   = [(n,m,pi,po,mt) for n,m,pi,po,mt,_,_ in alive]
_MID2PIN = {m: PIN.get(n) for n,m,*_ in MODELS}
_MID2SVC = {mid: svc for _, mid, _, _, _, _, svc in alive}   # openrouter or hf
json.dump(PIN, open(PROC/"provider_pins.json","w"), indent=2)
print(f"\npins applied; {len(MODELS)} models ready")



def call(mid, user, max_tok, web=False, timeout=180):
    """One question, one call. Routes to OpenRouter or HuggingFace, pins the
    provider where a pin exists, retries empty completions."""
    if mid in DEAD:
        return dict(text=None,usage={},http=403,error="model blocked",prov=None)
    svc = _MID2SVC.get(mid, "openrouter")
    body = {"model":mid, "temperature":TEMP, "max_tokens":max_tok,
            "messages":[{"role":"system","content":PR["system"]},
                        {"role":"user","content":user}]}
    if web and svc == "openrouter":
        body["plugins"] = [{"id":"web","max_results":3}]
    pin = _MID2PIN.get(mid)
    if pin and svc == "openrouter":
        body["provider"] = {"order":[pin], "allow_fallbacks":False}
    last_empty = None
    for att in range(5):
        if STOP.is_set():
            return dict(text=None,usage={},http=0,error="stopped",prov=None)
        try:
            r = _post_any(body, timeout, svc)
            if r.status_code == 200:
                d = r.json(); ch = d["choices"][0]
                txt = ch["message"].get("content")
                if not txt:
                    last_empty = dict(text=None, usage=d.get("usage",{}), http=204,
                                      error=f"empty ({ch.get('finish_reason')})",
                                      prov=d.get("provider", svc))
                    time.sleep(2+att); continue
                return dict(text=txt, usage=d.get("usage",{}), http=200,
                            error=None, prov=d.get("provider", svc))
            if r.status_code in (429,503,529):
                time.sleep(min(5*(1.8**att),60)); continue
            if r.status_code in (401,402):          # bad key / no credit — affects all
                STOP.set()
                return dict(text=None,usage={},http=r.status_code,
                            error=r.text[:200],prov=None)
            if r.status_code == 403:                # blocked for THIS model only
                with LOCK:
                    if mid not in DEAD:
                        DEAD.add(mid)
                        print(f"\n  *** {mid} BLOCKED (403) — skipping ***")
                        print(f"      {r.text[:160]}")
                return dict(text=None,usage={},http=403,
                            error=r.text[:200],prov=None)
            if r.status_code == 403:
                with LOCK:
                    if mid not in DEAD:
                        DEAD.add(mid)
                        print(f"\n  *** {mid} BLOCKED (403) — skipping ***")
                        print(f"      {r.text[:160]}")
                return dict(text=None,usage={},http=403,error=r.text[:200],prov=None)
            return dict(text=None,usage={},http=r.status_code,
                        error=r.text[:200],prov=None)
        except Exception:
            time.sleep(min(2**att,20))
    return last_empty or dict(text=None,usage={},http=0,
                              error="retries exhausted",prov=None)

def run(name, mid, pin_price, pout, max_tok, items, arm, swap=False, web=False):
    """Ask one model every question in `items`. Writes every attempt to disk."""
    tag  = f"{VERSION}__{name}__{arm}"
    path = RES/f"{tag}.jsonl"
    done = set()
    if path.exists():
        rows = [json.loads(l) for l in path.open(encoding="utf-8") if l.strip()]
        done = {r["item_id"] for r in rows if r.get("http")==200 and r.get("raw")}
    todo = items[~items.item_id.isin(done)]
    if not len(todo):
        print(f"  {tag}: complete ({len(done)})"); return
    print(f"  {tag}: {len(todo)} to do", flush=True)
    t0, n, bad = time.time(), [0], [0]
    def one(d, f):
        if STOP.is_set(): return
        res = call(mid, render(d, swap), max_tok, web)
        parsed, tier = parse(res["text"]) if res["http"]==200 else (None,"failed")
        u = res.get("usage") or {}
        cost = (u.get("prompt_tokens",0)/1e6*pin_price
                + u.get("completion_tokens",0)/1e6*pout)
        rec = dict(item_id=d["item_id"], benchmark_sha=SHA, model=name, arm=arm,
            model_id=mid, provider=res["prov"], pinned=_MID2PIN.get(mid),
            swap=bool(swap), web=bool(web),
            source=d["source"], target=d.get("target"),
            condition=d["condition"], truth=d["answer"],
            protein_usually=d.get("protein_usually"), group=d.get("group"),
            n_refs=d.get("n_refs"), is_decoy=bool(d.get("is_decoy", False)),
            pred=normalize(parsed), confidence=(parsed or {}).get("confidence"),
            rationale=(parsed or {}).get("rationale"), parse_tier=tier,
            raw=res["text"], prompt_tokens=u.get("prompt_tokens"),
            completion_tokens=u.get("completion_tokens"), cost_usd=cost,
            http=res["http"], error=res["error"])
        with LOCK:
            SPEND["usd"] += cost; n[0] += 1
            if res["http"] != 200: bad[0] += 1
            f.write(json.dumps(rec)+"\n"); f.flush()
            if SPEND["usd"] > BUDGET:
                STOP.set(); print(f"\n*** BUDGET ${BUDGET} REACHED — STOPPING ***")
            if n[0] % max(1,len(todo)//5) == 0 or n[0]==len(todo):
                print(f"    {n[0]}/{len(todo)}  {(time.time()-t0)/60:.1f}m  "
                      f"fail {bad[0]}  ${SPEND['usd']:.3f}", flush=True)
    with open(path,"a",encoding="utf-8") as f:
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            list(ex.map(lambda r: one(r[1].to_dict(), f), todo.iterrows()))
    print(f"    done {(time.time()-t0)/60:.1f}m  failures {bad[0]}/{len(todo)}")

json.dump(PIN, open(PROC/"provider_pins.json","w"), indent=2)

# ==================== 5. VERIFY ==========================================
print("\n"+"="*88); print("5. VERIFY THE PINS HOLD"); print("="*88)
bad_pin = 0
for name, mid, pi, po, mt in MODELS:
    want, got = PIN.get(name), set()
    for _, it in FULL.head(3).iterrows():
        r = call(mid, render(it.to_dict()), mt, timeout=120)
        if r["prov"]: got.add(r["prov"])
    if want:
        ok = got <= {want}
        bad_pin += (not ok)
        print(f"  [{'OK ' if ok else 'BAD'}] {name:<20}want={want:<14}got={got}")
    else:
        print(f"  [ -- ] {name:<20}unpinned{'':<10}got={got}"
              + ("" if len(got)<=1 else "   *** MULTI-PROVIDER — disclose ***"))
assert bad_pin == 0, "a pin was not honoured"

print("\n"+"="*88); print("ALREADY COLLECTED?"); print("="*88)
have = list(RES.glob("*.jsonl"))
if have:
    A = pd.concat([pd.read_json(p, lines=True) for p in have], ignore_index=True)
    A = A[A.http==200]
    print(f"  {len(A):,} rows on disk, {A.model.nunique()} models")
    stale = []
    for m, g in A.groupby("model"):
        want = PIN.get(m); provs = set(g.provider.dropna())
        flag = ""
        if want and provs - {want}: flag = f"  *** not all {want} ***"; stale.append(m)
        elif len(provs) > 1: flag = "  *** multiple providers ***"
        print(f"    {m:<20}{dict(g.provider.value_counts())}{flag}")
    if stale:
        print(f"\n  wrong-provider rows for {stale} — delete those files to recollect")
    todo_models = [n for n,*_ in MODELS if n not in set(A.model)]
    print(f"\n  models still to collect: {todo_models}")
else:
    print("  results folder empty")

# ---- projection -----------------------------------------------------------
print("\n"+"="*88); print("PLAN"); print("="*88)
n_calls = len(FULL) + len(USUAL)
tot = 0
print(f"{'model':<20}{'pin':<14}{'main+usually $':>16}{'+swap $':>10}")
print("-"*62)
for name, mid, pi, po, mt in MODELS:
    c = n_calls*(160/1e6*pi + 350/1e6*po)
    s = len(MAIN_ONLY)*(160/1e6*pi + 350/1e6*po)
    tot += c + s
    print(f"{name:<20}{str(PIN.get(name) or '—'):<14}{c:>16.3f}{s:>10.3f}")
web = len(WEB_ITEMS)*3*4/1000 + len(WEB_ITEMS)*(160/1e6*.1875 + 480/1e6*.9375)
print(f"\n  all models (new work only will actually be charged): ${tot:.2f}")
print(f"  web arm (already collected)                        : ${web:.2f}")
print(f"  budget ${BUDGET}   |   spent so far this session ${SPEND['usd']:.3f}")
print("\nReady. Run 03b (preflight) then 03c (collection).")

benchmark 336 main+decoy | 108 usually | sha=99be1e9522e4f933
  {'decoy': np.int64(120), 'habit_agrees': np.int64(108), 'habit_misleads': np.int64(108)}
parser self-test: 6/6 pass
HuggingFace token: present

2. MODEL ID CHECK  (1 call each)
  [OK ] gpt-oss-120b        openrouter openai/gpt-oss-120b                           ''
  [OK ] qwen3-32b           openrouter qwen/qwen3-32b                                ''
  [OK ] gemini-3.7-flash    openrouter google/gemini-3.7-flash                       ''
  [OK ] gpt-5.6-luna        openrouter openai/gpt-5.6-luna                           'ok'
  [OK ] gpt-5.6-terra       openrouter openai/gpt-5.6-terra                          'ok'
  [OK ] claude-haiku-4.5    openrouter anthropic/claude-haiku-4.5                    'ok'
  [OK ] llama-3.3-70b       openrouter meta-llama/llama-3.3-70b-instruct             'ok'
  [OK ] mistral-large       openrouter mistralai/mistral-large                       'ok'
  [OK ] deepseek-v3         openrouter deepse

In [ ]:
# ============================================================================
# 03b — PREFLIGHT  (~2 cents; verifies everything before the real run)
# ============================================================================
MINI  = pd.concat([FULL[FULL.condition==c].head(3)
                   for c in ["habit_agrees","habit_misleads","decoy"]]).reset_index(drop=True)
MINIU = USUAL.head(3)
print(f"preflight: {len(MINI)} main + {len(MINIU)} usually per model "
      f"= ~{(len(MINI)+len(MINIU))*len(MODELS)+8} calls\n")

SPEND["usd"] = 0.0; STOP.clear()
rows = []

def probe(name, mid, pin, pout, mt, item, swap=False, web=False):
    t0 = time.time()
    res = call(mid, render(item, swap), mt, web, timeout=180)
    parsed, tier = parse(res["text"]) if res["http"]==200 else (None,"failed")
    pred = normalize(parsed)
    u = res.get("usage") or {}
    cost = u.get("prompt_tokens",0)/1e6*pin + u.get("completion_tokens",0)/1e6*pout
    with LOCK: SPEND["usd"] += cost
    return dict(model=name, cond=item["condition"], swap=swap, web=web,
        source=item["source"], target=item.get("target"), truth=item["answer"],
        pred=pred, correct=(pred==item["answer"]),
        conf=(parsed or {}).get("confidence"), tier=tier,
        rationale=(parsed or {}).get("rationale"),
        raw=(res["text"] or "")[:200], http=res["http"], err=res["error"],
        prov=res["prov"], out_tok=u.get("completion_tokens"),
        secs=time.time()-t0, cost=cost)

print("="*96); print("A. EVERY MODEL, EVERY QUESTION TYPE"); print("="*96)
for name, mid, pin, pout, mt in MODELS:
    print(f"\n{name}", flush=True)
    for _, it in pd.concat([MINI, MINIU]).iterrows():
        r = probe(name, mid, pin, pout, mt, it.to_dict()); rows.append(r)
        ok = r["http"]==200 and r["pred"]
        print(f"  [{'OK ' if ok else 'BAD'}] {r['cond']:<15}"
              f"{str(r['source']):>9}->{str(r['target'] or '—'):<9} "
              f"truth={str(r['truth']):<16} pred={str(r['pred']):<16}"
              f"{r['tier']:<9}{r['secs']:>5.1f}s"
              + (f"  ERR {str(r['err'])[:55]}" if r["http"]!=200 else ""))

print("\n"+"="*96); print("B. ORDER-SWAP — does flipping the options change the answer?")
print("="*96)
for name, mid, pin, pout, mt in MODELS[:3]:
    it = MINI[MINI.condition=="habit_agrees"].iloc[0].to_dict()
    a = probe(name, mid, pin, pout, mt, it, swap=False)
    b = probe(name, mid, pin, pout, mt, it, swap=True)
    rows += [a,b]
    print(f"  {name:<20} normal={str(a['pred']):<14} swapped={str(b['pred']):<14}"
          f"  {'same' if a['pred']==b['pred'] else 'DIFFERENT'}")

print("\n"+"="*96); print("C. WEB SEARCH — is retrieval actually happening?"); print("="*96)
for _, it in MINI.head(2).iterrows():
    r = probe("gemini-3.7-flash","google/gemini-3.7-flash",.1875,.9375,3000,
              it.to_dict(), web=True); rows.append(r)
    print(f"  [{'OK ' if r['http']==200 else 'BAD'}] {r['cond']:<15}"
          f"{r['source']}->{r['target']}  pred={str(r['pred']):<16}"
          f"{r['secs']:>5.1f}s  ${r['cost']:.4f}  {r['out_tok']} tok")
    print(f"        {str(r['rationale'])[:130]}")

T  = pd.DataFrame(rows)
ok = T[T.http==200]
print("\n"+"="*96); print("D. DID ANYTHING BREAK?"); print("="*96)
checks = [
 ("every model responded", T.groupby('model').http.apply(lambda s:(s==200).any()).all()),
 ("no HTTP failures",      (T.http==200).all()),
 ("everything parsed",     ok.pred.notna().all()),
 ("decoys answered",       len(ok[ok.cond=='decoy'])>0),
 ("'usually' probes work", len(ok[ok.cond=='usually'])>0),
 ("swap arm works",        ok[ok.swap].pred.notna().all() if ok.swap.any() else True),
 ("web arm works",         ok[ok.web].pred.notna().all() if ok.web.any() else True),
]
for n_,c in checks: print(f"  [{'PASS' if c else 'FAIL'}] {n_}")
if (T.http!=200).any():
    print("\n  FAILURES:")
    print("  "+T[T.http!=200][["model","cond","http","err"]].to_string(index=False).replace("\n","\n  "))
if ok.pred.isna().any():
    print("\n  UNPARSED — raw text:")
    for _, r in ok[ok.pred.isna()].head(3).iterrows():
        print(f"    {r.model}/{r.cond}: {r.raw!r}")

print("\n"+"="*96); print("E. SANITY (3 items each — machinery check, not a result)")
print("="*96)
print(ok.pivot_table(index="model", columns="cond", values="correct",
                     aggfunc="mean").round(2).to_string())
d = ok[ok.cond=="decoy"]
print("\ndecoy refusal rate (some models should be >0):")
print("  "+(d.assign(ref=d.pred=="not_established").groupby("model").ref.mean()
      .round(2).to_string().replace("\n","\n  ")))

print("\n"+"="*96); print("F. READ THESE RATIONALES"); print("="*96)
for m in ok.model.unique():
    s = ok[(ok.model==m)&(ok.cond=="habit_misleads")&(~ok.web)]
    if not len(s): continue
    r = s.iloc[0]
    print(f"\n  {m}  |  {r.source} -> {r.target}  (truth: {r.truth})")
    print(f"    said {r.pred}, confidence {r.conf}")
    print(f"    {str(r.rationale)[:150]}")

print("\n"+"="*96); print("G. COST PROJECTION"); print("="*96)
print(f"  preflight cost: ${SPEND['usd']:.4f} over {len(T)} calls\n")
per = ok[~ok.web].groupby("model").agg(secs=("secs","mean"), cost=("cost","mean"),
                                       out=("out_tok","mean"))
n_calls = len(FULL)+len(USUAL)
print(f"{'model':<20}{'s/call':>8}{'out tok':>9}{'$/call':>9}{'full run $':>12}{'min':>7}")
print("-"*66)
tot = 0
for m in per.index:
    c = per.loc[m,"cost"]*n_calls; tot += c
    print(f"{m:<20}{per.loc[m,'secs']:>8.1f}{per.loc[m,'out']:>9.0f}"
          f"{per.loc[m,'cost']:>9.4f}{c:>12.3f}"
          f"{per.loc[m,'secs']*n_calls/WORKERS/60:>7.0f}")
web_cost = len(FULL)*3*4/1000 + len(FULL)*(160/1e6*.1875 + 400/1e6*.9375)
print(f"\n  main arms        : ${tot:.2f}")
print(f"  swap (3 models)  : ${tot*0.13:.2f}")
print(f"  web (Gemini, x3) : ${web_cost:.2f}")
print(f"  PROJECTED TOTAL  : ${tot + tot*0.13 + web_cost:.2f}   (budget ${BUDGET})")

good = all(c for _,c in checks)
print("\n"+"="*96)
print("ALL CHECKS PASSED — cell 03c is safe to run" if good
      else "*** SOMETHING FAILED — do not run 03c ***")
print("="*96)

preflight: 9 main + 3 usually per model = ~68 calls

A. EVERY MODEL, EVERY QUESTION TYPE

gpt-oss-120b
  [OK ] habit_agrees       NFIL3->PTGS2     truth=inhibits         pred=inhibits        strict    18.7s
  [OK ] habit_agrees        DOK1->ITGB2     truth=inhibits         pred=not_established strict     5.3s
  [OK ] habit_agrees       HDAC1->ABCB1     truth=inhibits         pred=inhibits        strict     2.2s
  [OK ] habit_misleads     NFIL3->IL3       truth=stimulates       pred=inhibits        strict     3.1s
  [OK ] habit_misleads      DOK1->RASGRP3   truth=stimulates       pred=not_established strict     3.9s
  [OK ] habit_misleads     HDAC1->COL1A2    truth=stimulates       pred=inhibits        strict     6.9s
  [OK ] decoy             DNMT3B->OXT       truth=not_established  pred=not_established strict     3.1s
  [OK ] decoy             DNMT3B->STAT3     truth=not_established  pred=not_established strict     9.7s
  [OK ] decoy              MAPK7->HIPK2     truth=not_established

In [ ]:
# ============================================================================
# 03c — FULL COLLECTION   *** THIS IS THE CELL THAT SPENDS MONEY ***
# ----------------------------------------------------------------------------
# Four arms per model:
#   main     336 questions (216 real + 120 fabricated pairs)
#   usually  108 probes, asked separately, on each protein's general behaviour
#   swap     216 questions with the answer options listed in reverse
#   web      216 questions with web search on (Gemini and Terra only)
#
# Resumable and safe to interrupt: re-running skips anything already answered.
# A 403 skips that one model; 401/402 halts everything.
# Roughly $6 for all 13 models. Run 03b first.
assert "WEB_ITEMS" in globals(), "run the patch cell first"
SPEND["usd"] = 0.0; STOP.clear()
RUN_T0 = time.time()

print("="*70); print("MAIN ARM — 5 models x (336 questions + 108 usually)")
print("="*70)
for name, mid, pin, pout, mt in MODELS:
    if STOP.is_set(): print("\n*** HALTED ***"); break
    print(f"\n{name}", flush=True)
    run(name, mid, pin, pout, mt, FULL,  "main")
    run(name, mid, pin, pout, mt, USUAL, "usually")


print("\n"+"="*70); print("ORDER-SWAP ARM — all models"); print("="*70)
for name, mid, pin_, pout, mt in MODELS:          # was MODELS[:3]
    if STOP.is_set(): break
    run(name, mid, pin_, pout, mt, MAIN_ONLY, "swap", swap=True)
print("\n"+"="*70); print("WEB-SEARCH ARM"); print("="*70)
for name, mid, pin_, pout, mt in MODELS:
    if STOP.is_set(): break
    if name not in WEB_MODELS: continue
    if _MID2SVC.get(mid) != "openrouter":
        print(f"  {name}: web plugin is OpenRouter-only, skipping"); continue
    run(name, mid, pin_, pout, mt, WEB_ITEMS, "web", web=True)

# ---- inventory -----------------------------------------------------------
print("\n"+"="*70); print("INVENTORY"); print("="*70)
A = pd.concat([pd.read_json(p, lines=True) for p in RES.glob("*.jsonl")],
              ignore_index=True)
print(f"rows {len(A):,}   spent ${SPEND['usd']:.2f}   "
      f"wall clock {(time.time()-RUN_T0)/60:.0f} min")
print(f"http failures {(A.http!=200).mean():.2%}   "
      f"unparsed {(A[A.http==200].pred.isna()).mean():.2%}")

print("\nunique items per model x arm:")
print(A[A.http==200].pivot_table(index="model", columns="arm",
      values="item_id", aggfunc="nunique", fill_value=0).to_string())

print("\nEXPECTED:  main 336 | usually 108 | swap 216 (all 5 models) | web 216 (Gemini)")



if (A.http!=200).any():
    print("\nfailures by model and status:")
    print(A[A.http!=200].groupby(["model","http"]).size().to_string())
if (A[A.http==200].pred.isna()).any():
    u = A[(A.http==200) & A.pred.isna()]
    print(f"\nunparsed ({len(u)} rows):")
    print(u.groupby("model").size().to_string())
    for _, r in u.head(3).iterrows():
        print(f"    {r.model}: {str(r.raw)[:120]!r}")

print("\nspend by model:")
print(A.groupby("model").cost_usd.sum().round(3).to_string())

# ---- first look at the headline ------------------------------------------
print("\n"+"="*70); print("FIRST LOOK  (main arm, real questions only)"); print("="*70)
M = A[(A.arm=="main") & (A.http==200) & (~A.is_decoy) & A.pred.notna()]
if len(M):
    t = M.assign(ok=M.pred==M.truth).pivot_table(
        index="model", columns="condition", values="ok", aggfunc="mean")
    if {"habit_agrees","habit_misleads"} <= set(t.columns):
        t["gap_pp"] = ((t.habit_agrees - t.habit_misleads)*100).round(1)
        print((t[["habit_agrees","habit_misleads"]]*100).round(1)
              .assign(gap_pp=t.gap_pp).to_string())
        print("\n  gap = how much worse models do when the protein does the")
        print("  opposite of its usual thing. This is the whole study.")

MAIN ARM — 5 models x (336 questions + 108 usually)

gpt-oss-120b
  v2__gpt-oss-120b__main: complete (336)
  v2__gpt-oss-120b__usually: complete (108)

qwen3-32b
  v2__qwen3-32b__main: complete (336)
  v2__qwen3-32b__usually: complete (108)

gemini-3.7-flash
  v2__gemini-3.7-flash__main: complete (336)
  v2__gemini-3.7-flash__usually: complete (108)

gpt-5.6-luna
  v2__gpt-5.6-luna__main: complete (336)
  v2__gpt-5.6-luna__usually: complete (108)

gpt-5.6-terra
  v2__gpt-5.6-terra__main: complete (336)
  v2__gpt-5.6-terra__usually: complete (108)

claude-haiku-4.5
  v2__claude-haiku-4.5__main: complete (336)
  v2__claude-haiku-4.5__usually: complete (108)

llama-3.3-70b
  v2__llama-3.3-70b__main: complete (336)
  v2__llama-3.3-70b__usually: complete (108)

mistral-large
  v2__mistral-large__main: 122 to do
    24/122  0.2m  fail 0  $0.015
    48/122  0.3m  fail 0  $0.030
    72/122  0.4m  fail 0  $0.044
    96/122  0.6m  fail 0  $0.059
    120/122  0.7m  fail 0  $0.074
    122/122  0.7

In [ ]:
# ============================================================================
# 03d — COLLECT THE NEW MODELS
# ----------------------------------------------------------------------------
#   gemma-4-31b   newest Gemma, alongside gemma-3-27b and medgemma-27b
#   llama-3-70b   base for the second biomedical pair (HF; may be asleep)
#   med42-70b     Med42-v2, instruct + preference tuned FROM Llama-3-70B
#
# Two independent biomedical comparisons, by different adaptation methods:
#   MedGemma-27B vs Gemma-3-27B   continued pretraining on biomedical text
#   Med42-70B    vs Llama-3-70B   clinical instruct + preference tuning
#
# Skips models that failed the ID check rather than refusing to run.
# Nothing already collected is touched. ~$0.10 total.
# ============================================================================
WANTED    = ["gemma-4-31b", "llama-3-70b", "med42-70b"]
AVAILABLE = {n for n, *_ in MODELS}
NEW       = [m for m in WANTED if m in AVAILABLE]
SKIPPED   = [m for m in WANTED if m not in AVAILABLE]

print(f"collecting: {NEW}")
if SKIPPED:
    print(f"SKIPPED (failed the ID check in 03a): {SKIPPED}")
    print("  A 503 means the HuggingFace endpoint was asleep, not missing.")
    print("  Wake it, re-run 03a, then re-run this cell to fill the gap.\n")
assert NEW, "nothing to collect"

SPEND["usd"] = 0.0; STOP.clear(); DEAD.clear()
T0 = time.time()

SLOW = {"llama-3-70b", "med42-70b"}       # single-threaded HF endpoints
_W = WORKERS
for name, mid, pi, po, mt in MODELS:
    if name not in NEW: continue
    if STOP.is_set(): print("\n*** HALTED ***"); break
    globals()["WORKERS"] = 3 if name in SLOW else _W
    print(f"\n{name}   [{_MID2SVC.get(mid)}]  pin={PIN.get(name) or '—'}  "
          f"workers={WORKERS}", flush=True)
    run(name, mid, pi, po, mt, FULL,      "main")
    run(name, mid, pi, po, mt, USUAL,     "usually")
    run(name, mid, pi, po, mt, MAIN_ONLY, "swap", swap=True)
globals()["WORKERS"] = _W

# ---- inventory -----------------------------------------------------------
print("\n"+"="*80); print("INVENTORY"); print("="*80)
A  = pd.concat([pd.read_json(p, lines=True) for p in RES.glob("*.jsonl")],
               ignore_index=True)
ok = A[A.http==200]
print(f"OpenRouter spend this session ${SPEND['usd']:.3f}   "
      f"{(time.time()-T0)/60:.0f} min   {ok.model.nunique()} models total")
print("\nunique items per model x arm:")
print(ok.pivot_table(index="model", columns="arm", values="item_id",
                     aggfunc="nunique", fill_value=0).to_string())
print("\nEXPECTED per model:  main 336 | usually 108 | swap 216")

bad = A[(A.http!=200) & A.model.isin(NEW)]
if len(bad):
    print("\nfailures among the new models:")
    print(bad.groupby(["model","http"]).size().to_string())
unp = ok[ok.model.isin(NEW) & ok.pred.isna()]
if len(unp):
    print(f"\nunparsed ({len(unp)}):"); print(unp.groupby("model").size().to_string())
    for _, r in unp.head(3).iterrows():
        print(f"    {r.model}: {str(r.raw)[:110]!r}")

# ---- results -------------------------------------------------------------
M = ok[(ok.arm=="main") & (~ok.is_decoy) & ok.pred.notna()]
D = ok[(ok.arm=="main") &   ok.is_decoy]
def _g(m):
    g = M[M.model==m]
    if not len(g): return None
    a = (g[g.condition=="habit_agrees"].pred == g[g.condition=="habit_agrees"].truth).mean()
    b = (g[g.condition=="habit_misleads"].pred == g[g.condition=="habit_misleads"].truth).mean()
    return a, b, (a-b)*100

print("\n"+"="*80); print("THE GAP — NEW MODELS"); print("="*80)
print(f"{'model':<16}{'usual':>8}{'exception':>11}{'gap':>7}{'abstains':>10}"
      f"{'refuses fakes':>15}")
print("-"*70)
for m in NEW:
    r = _g(m)
    if not r: print(f"{m:<16}  (no data)"); continue
    g, d = M[M.model==m], D[D.model==m]
    print(f"{m:<16}{r[0]:>8.1%}{r[1]:>11.1%}{r[2]:>7.1f}"
          f"{(g.pred=='not_established').mean():>10.1%}"
          f"{(d.pred=='not_established').mean() if len(d) else float('nan'):>15.1%}")

print("\n"+"="*80); print("BIOMEDICAL PAIRS — same base, same version, same size")
print("="*80)
for lab, base, ft in [("continued pretraining",                "gemma-3-27b", "medgemma-27b"),
                      ("clinical instruct + preference tuning","llama-3-70b", "med42-70b")]:
    print(f"\n  {lab}")
    res = {}
    print(f"    {'model':<16}{'usual':>8}{'exception':>11}{'gap':>7}{'refuses fakes':>15}")
    for m in (base, ft):
        r = _g(m)
        if not r: print(f"    {m:<16}  (not collected)"); continue
        res[m] = r; d = D[D.model==m]
        print(f"    {m:<16}{r[0]:>8.1%}{r[1]:>11.1%}{r[2]:>7.1f}"
              f"{(d.pred=='not_established').mean() if len(d) else float('nan'):>15.1%}")
    if base in res and ft in res:
        print(f"    {'CHANGE':<16}{'':>8}{'':>11}{res[ft][2]-res[base][2]:>+7.1f}")
    else:
        print("    pair incomplete — cannot compare")
print("\n  Positive change = biomedical training WIDENED the shortcut.")

print("\n"+"="*80); print("GEMMA FAMILY — does a newer generation help?"); print("="*80)
print(f"{'model':<16}{'usual':>8}{'exception':>11}{'gap':>7}")
print("-"*44)
for m in ["gemma-3-27b","gemma-4-31b","medgemma-27b"]:
    r = _g(m)
    if r: print(f"{m:<16}{r[0]:>8.1%}{r[1]:>11.1%}{r[2]:>7.1f}")

print(f"\n{'='*80}\n{ok.model.nunique()} models collected.")
if SKIPPED: print(f"still missing: {SKIPPED}")
print(f"Re-run notebook 04.\n{'='*80}")

AssertionError: not in MODELS_SPEC (or failed the ID check): ['llama-3-70b']

DIAGNOSTICS

In [60]:
# ============================================================================
# TEMP — TEST THE THREE CANDIDATES  (delete after)
# ============================================================================
import requests, os, time, json
HF_KEY = os.environ.get("HF_TOKEN",""); OR_KEY = os.environ["OPENROUTER_API_KEY"]
HF_URL = "https://router.huggingface.co/v1/chat/completions"
OR_URL = "https://openrouter.ai/api/v1/chat/completions"

CAND = [
 ("gemma-4-31b",  "google/gemma-4-31b-it",                         "or"),
 ("llama-3-70b",  "meta-llama/Meta-Llama-3-70B-Instruct:featherless-ai", "hf"),
 ("med42-70b",    "m42-health/Llama3-Med42-70B:featherless-ai",     "hf"),
]
SYS = ("You are an expert molecular biologist. Reply with a single JSON object "
       "and nothing else. No preamble, no markdown, no code fences.")
Q = ("What is the net regulatory effect of AKT1 on GSK3B at the protein level?\n"
     "Answer stimulates if the net effect is activating, inhibits if repressing, "
     "or not_established if no such direct relationship is documented.\n\n"
     "Respond with JSON only:\n"
     '{"answer": "stimulates" | "inhibits" | "not_established", '
     '"confidence": <integer 1-10>, "rationale": "<one sentence>"}')

def hit(mid, svc, tries=3):
    url, key = (HF_URL, HF_KEY) if svc=="hf" else (OR_URL, OR_KEY)
    ok, secs, outs, texts, codes = 0, [], [], [], []
    for _ in range(tries):
        t0 = time.time()
        try:
            r = requests.post(url, headers={"Authorization":f"Bearer {key}"},
                json={"model":mid,"temperature":0.3,"max_tokens":400,
                      "messages":[{"role":"system","content":SYS},
                                  {"role":"user","content":Q}]}, timeout=240)
            codes.append(r.status_code)
            if r.status_code==200:
                d = r.json(); t = d["choices"][0]["message"].get("content") or ""
                if t:
                    ok += 1; secs.append(time.time()-t0); texts.append(t)
                    outs.append(d.get("usage",{}).get("completion_tokens",0))
            elif r.status_code in (503,429):
                time.sleep(20)
        except Exception as e:
            codes.append(type(e).__name__)
    return ok, secs, outs, texts, codes

print("="*92); print("CAN WE RUN THESE?"); print("="*92)
GOOD = []
for name, mid, svc in CAND:
    ok, secs, outs, texts, codes = hit(mid, svc)
    if not ok:
        print(f"  [BAD] {name:<14}{svc:<4}{codes}"); continue
    s, o = sum(secs)/len(secs), sum(outs)/len(outs)
    try:    parsed = sum(bool(normalize(parse(t)[0])) for t in texts)
    except NameError: parsed = sum('"answer"' in t for t in texts)
    print(f"  [OK ] {name:<14}{svc:<4}{ok}/3 responded, {parsed}/3 parsed, "
          f"{s:>5.1f}s, {o:>4.0f} out tok")
    print(f"        {texts[0].strip()[:110]!r}")
    GOOD.append((name, mid, svc, s, o))

print("\n"+"="*92); print("COST AND TIME"); print("="*92)
print("Gemma 4 gets all three arms (660 calls). The 70B pair gets main +")
print("usually only (444 calls) — the swap arm is a validity check already")
print("run on 13 models, and these endpoints are slow.\n")
print(f"{'model':<14}{'calls':>7}{'s/call':>8}{'hours':>7}{'est $':>8}  billed to")
print("-"*62)
for name, mid, svc, s, o in GOOD:
    n = 660 if svc=="or" else 444
    workers = 6 if svc=="or" else 2          # HF 70B endpoints are single-threaded
    if name=="gemma-4-31b":  c = n*(160/1e6*0.12 + o/1e6*0.35)
    else:                    c = n*(160/1e6*0.30 + o/1e6*0.60)   # HF 70B, approx
    print(f"{name:<14}{n:>7}{s:>8.1f}{s*n/workers/3600:>7.1f}{c:>8.2f}  "
          f"{'OpenRouter' if svc=='or' else 'HuggingFace'}")
hf_tot = sum(444*(160/1e6*0.30 + o/1e6*0.60) for n_,m_,sv,s_,o in GOOD if sv=="hf")
print(f"\n  HuggingFace total ~${hf_tot:.2f}  (budget $1.80)")
print("  Watch the first progress line during the real run and stop if the")
print("  spend looks wrong — HF provider pricing varies.")

CAN WE RUN THESE?
  [OK ] gemma-4-31b   or  3/3 responded, 3/3 parsed,   1.8s,   44 out tok
        '{"answer": "inhibits", "confidence": 10, "rationale": "AKT1 phosphorylates GSK3B at Serine 9, which leads to t'
  [OK ] llama-3-70b   hf  1/3 responded, 1/3 parsed,  30.6s,   38 out tok
        '{"answer": "inhibits", "confidence": 9, "rationale": "AKT1 phosphorylates and inhibits GSK3B, thereby reducing'
  [OK ] med42-70b     hf  3/3 responded, 3/3 parsed,   9.2s,   47 out tok
        '{ "answer": "inhibits", "confidence": 9, "rationale": "AKT1 phosphorylates GSK3B at Ser9, leading to its inact'

COST AND TIME
Gemma 4 gets all three arms (660 calls). The 70B pair gets main +
usually only (444 calls) — the swap arm is a validity check already
run on 13 models, and these endpoints are slow.

model           calls  s/call  hours   est $  billed to
--------------------------------------------------------------
gemma-4-31b       660     1.8    0.1    0.02  OpenRouter
llama-3-70b       444  

In [ ]:
# ============================================================================
# TEMP — WAKE UP LLAMA-3-70B  (delete after)
# ============================================================================
import requests, os, time
MID = "meta-llama/Meta-Llama-3-70B-Instruct:featherless-ai"
HDR = {"Authorization": f"Bearer {os.environ['HF_TOKEN']}"}
URL = "https://router.huggingface.co/v1/chat/completions"

for att in range(6):
    t0 = time.time()
    r = requests.post(URL, headers=HDR, json={
        "model": MID, "max_tokens": 32, "temperature": 0,
        "messages": [{"role":"user","content":"Reply with one word: ok"}]},
        timeout=300)
    el = time.time()-t0
    if r.status_code == 200:
        txt = (r.json()["choices"][0]["message"].get("content") or "").strip()
        print(f"  attempt {att+1}: AWAKE in {el:.0f}s -> {txt[:30]!r}")
        break
    print(f"  attempt {att+1}: {r.status_code} after {el:.0f}s  {r.text[:90]}")
    time.sleep(45)
else:
    print("\n  still asleep after 6 tries over ~5 min.")
    print("  Options: try again later, or drop it and note that the second")
    print("  biomedical pair is unavailable.")

  attempt 1: 503 after 15s  {"error":{"message":"meta-llama/Meta-Llama-3-70B-Instruct is temporarily at capacity. Plea
  attempt 2: 503 after 15s  {"error":{"message":"meta-llama/Meta-Llama-3-70B-Instruct is temporarily at capacity. Plea
